# Personi B - Electronic Online Shop Machine Learning Project

This notebook implements all assigned Personi B tasks:

1. Logistic Regression for DemandLevel prediction
2. Neural Network Architecture 1
3. Neural Network Architecture 2
4. Neural Network Hyperparameter Tuning
5. K-Means Clustering
6. Elbow Method and Silhouette Score
7. Cluster comparison with real DemandLevel labels
8. Final model comparison table

The expected dataset path is `data/electronic_online_shop.csv`. If the official dataset is missing, the code creates a generated sample dataset for demonstration only.


## Setup

The project source code is stored in `src/pipeline_dyqani_elektronik.py`.
The notebook imports those reusable functions so the analysis stays organized and easy to rerun.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.pipeline_dyqani_elektronik import (
    TARGET_COLUMN,
    load_dataset,
    split_columns,
    classification_workflow,
    neural_network_grid_search,
    clustering_workflow,
)


## Data Loading and Preprocessing

The preprocessing stage detects numerical and categorical columns automatically.
Numerical variables are standardized, and categorical variables are one-hot encoded inside scikit-learn pipelines.


In [ ]:
df = load_dataset()
print("Dataset shape:", df.shape)
display(df.head())


In [ ]:
print("Target distribution:")
display(df[TARGET_COLUMN].value_counts().to_frame("Count"))

X, y, numeric_features, categorical_features = split_columns(df)
print("Numerical features:", numeric_features)
print("Categorical features:", categorical_features)


## Task 1 - Logistic Regression

Logistic Regression is trained with data preprocessing and GridSearchCV.
The same workflow also trains KNN and Decision Tree so Logistic Regression can be compared against both models.
Metrics calculated:

- Accuracy
- Precision
- Recall
- F1 Score
- Confusion Matrix


In [ ]:
classification_results, grid_models = classification_workflow(df)
display(classification_results)

# The Logistic Regression best parameters are available after fitting:
if "Logistic Regression" in grid_models:
    print("Best Logistic Regression parameters:")
    print(grid_models["Logistic Regression"].best_params_)


## Task 2 - Neural Network Architecture 1

Architecture 1 uses:

- Input layer from preprocessed features
- Hidden layer with 50 neurons
- Output layer for DemandLevel classes
- ReLU activation
- Adam optimizer

The architecture is intentionally simple, which makes it useful as a baseline neural network.


In [ ]:
display(classification_results[classification_results["Model"] == "Neural Network Architecture 1"])


## Task 3 - Neural Network Architecture 2

Architecture 2 uses two hidden layers:

- Hidden layer 1: 100 neurons
- Hidden layer 2: 50 neurons

This architecture can model more complex relationships, but it may require more training time and can overfit if the dataset is small.


In [ ]:
nn_comparison = classification_results[
    classification_results["Model"].isin([
        "Neural Network Architecture 1",
        "Neural Network Architecture 2",
    ])
]
display(nn_comparison)


## Task 4 - Neural Network Hyperparameter Tuning

GridSearchCV tunes:

- Hidden layers
- Learning rate
- Activation function
- Alpha regularization
- Batch size

The best model is selected by weighted F1 Score.


In [ ]:
nn_grid_results, best_nn_summary = neural_network_grid_search(df)
display(nn_grid_results[["rank_test_score", "mean_test_score", "std_test_score", "params"]].head(10))
display(best_nn_summary)


## Task 5 and 6 - K-Means Clustering, Elbow Method, and Silhouette Score

K-Means is applied after removing the target column `DemandLevel`.
The analysis tests K values from 2 to 6, then uses inertia and silhouette score to choose an appropriate cluster structure.


In [ ]:
cluster_metrics, cluster_label_crosstab, matching_percentage = clustering_workflow(df)
display(cluster_metrics)
print(f"Cluster-label matching percentage: {matching_percentage:.2f}%")


## Task 7 - Compare Clusters with Real Labels

This crosstab compares unsupervised K-Means clusters with the actual `DemandLevel` labels.
A high matching percentage suggests that the natural clusters are similar to the real demand categories.


In [ ]:
display(cluster_label_crosstab)


## Task 8 - Final Model Comparison

The final comparison table ranks all supervised models by weighted F1 Score.
The best model should be selected using both the ranking table and the confusion matrices.


In [ ]:
display(classification_results.sort_values("Rank"))
best_model = classification_results.sort_values("Rank").iloc[0]
print("Best model:", best_model["Model"])
print("Best weighted F1 Score:", round(best_model["F1 Score"], 4))


## Output Files

Running this notebook creates result tables and charts in:

- `outputs/model_comparison.csv`
- `outputs/neural_network_gridsearch_results.csv`
- `outputs/best_neural_network_configuration.csv`
- `outputs/cluster_metrics.csv`
- `outputs/cluster_label_crosstab.csv`
- `outputs/figures/`
